# 02 — Mechanistic ablation: MLP Neurons

Let's now try to ablate MLP neurons and see what happens.

In [1]:
try:
    # model loader
    from llm_ablation.models import load_model
    # model hook points
    from llm_ablation.hooks import (
        ablate_neurons, ablate_layer,
        collect_activations
    )
    # ablation utilities
    from llm_ablation.mech import (
        logit_diff,
        run_factual_recall_task_mlp,
        run_factual_recall_task_layer
    )
    # numpy
    import numpy as np
except Exception as e:
    raise(e)

# load synthetic data
from llm_ablation.data import (
    FACTUAL_RECALL_PAIRS,
    GENERAL_KNOWLEDGE_RECALL_PAIRS
)

# load model chekpoint
bundle = load_model()
print(f"{bundle.name}: {bundle.n_layers} layers x {bundle.n_heads} heads")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen/Qwen2.5-0.5B-Instruct: 24 layers x 14 heads


## Now we ablate at the MLP neuron level.

The model has `n_layers` layers, each with its MLP contribution. In the MLP layer, we are going to iteratively ablate each of the `in_features` neuron of the `down_proj` matrix and measure the ablation drop.

For the example model `Qwen2.5-0.5B-Instruct`, we have 24 layers with an inner dimension of 4864 input features.

Let's first try to run the ablation by selecting an arbitrary set of Layers and an anbitrary set of MLP Neurons.

In [2]:
import random

# select number of layers
N_LAYERS: int = 5
# number of neurons per layer
N_NEURONS: int = 15

# select parameters
LAYER: list = random.sample(
    range(0, bundle.n_layers-1),
    N_LAYERS
)
# hidden size
HIDDEN_SIZE: int = bundle.layer(LAYER[0]).mlp.down_proj.in_features
# neurons to ablate...
NEURONS: list = random.sample(
    range(0, HIDDEN_SIZE-1),
    N_NEURONS
)

print(f"Ablating {N_LAYERS} layers: {LAYER}")
print(f"Ablating Neurons {NEURONS} in every layer")

Ablating 5 layers: [6, 18, 0, 22, 11]
Ablating Neurons [1126, 4859, 4328, 3234, 2779, 4174, 3000, 3669, 166, 724, 4597, 4387, 4551, 942, 3255] in every layer


## Iteratively ablate selected neurons

In [ ]:
# calculate baseline logit diff with all neurons unchanged
baseline = np.array([logit_diff(bundle, p, c, i) for p, c, i in FACTUAL_RECALL_PAIRS])

# drops array (one row per layer)
drops = np.zeros((bundle.n_layers, HIDDEN_SIZE))

# sweep over layers
for layer in LAYER:
    # sweep
    for neuron in NEURONS:
        with ablate_neurons(bundle, layer, [neuron], mode="zero", reference_cache=None):
            # recompute logit diff with ablated head
            ablated = np.array([logit_diff(bundle, p, c, i) for p, c, i in FACTUAL_RECALL_PAIRS])
                    
        # measure and save ablation run result for this layer
        drops[layer, neuron] = (baseline - ablated).mean()

# print results
print(f"Received ablation results: {drops.shape} ndarray")

Received albation results: (24, 4864) ndarray


## Examine Results (Neuron Ablation)

In [4]:
# get logit diff from ablation matrix
def ablation_value(ablation_results, layer, neuron):
    return ablation_results[layer, neuron]

for layer in LAYER:
    for neuron in NEURONS:
        print(f"Layer {layer:2d}, Ablated Neuron: {neuron:2d} -> Logit Drop {ablation_value(drops, layer, neuron):2f}")

Layer  6, Ablated Neuron: 1126 -> Logit Drop 0.006250
Layer  6, Ablated Neuron: 4859 -> Logit Drop -0.004687
Layer  6, Ablated Neuron: 4328 -> Logit Drop 0.009375
Layer  6, Ablated Neuron: 3234 -> Logit Drop -0.003125
Layer  6, Ablated Neuron: 2779 -> Logit Drop 0.014063
Layer  6, Ablated Neuron: 4174 -> Logit Drop 0.001563
Layer  6, Ablated Neuron: 3000 -> Logit Drop 0.001563
Layer  6, Ablated Neuron: 3669 -> Logit Drop -0.003125
Layer  6, Ablated Neuron: 166 -> Logit Drop -0.004687
Layer  6, Ablated Neuron: 724 -> Logit Drop 0.010937
Layer  6, Ablated Neuron: 4597 -> Logit Drop 0.009375
Layer  6, Ablated Neuron: 4387 -> Logit Drop 0.004687
Layer  6, Ablated Neuron: 4551 -> Logit Drop -0.001563
Layer  6, Ablated Neuron: 942 -> Logit Drop 0.001563
Layer  6, Ablated Neuron: 3255 -> Logit Drop 0.050000
Layer 18, Ablated Neuron: 1126 -> Logit Drop 0.001563
Layer 18, Ablated Neuron: 4859 -> Logit Drop -0.020313
Layer 18, Ablated Neuron: 4328 -> Logit Drop -0.006250
Layer 18, Ablated Neuron

# Ablate entire Layers

This last experiment iteratively skips an entire transformer layer. Zero-mode means residual stream passes as-is (unchanged)

In [7]:
# calculate baseline logit diff with all neurons unchanged
baseline = np.array([logit_diff(bundle, p, c, i) for p, c, i in FACTUAL_RECALL_PAIRS])

# drop mesurement
layer_drop = np.zeros(bundle.n_layers)

# sweep over layers
for layer in range(bundle.n_layers):
    with ablate_layer(bundle=bundle, layer=layer, mode="zero", reference_cache=None):
        ablated = np.array([logit_diff(bundle, p, c, i) for p, c, i in FACTUAL_RECALL_PAIRS])

    # measure logit drop per layer
    layer_drop[layer] = (baseline - ablated).mean()

# print results
print(f"Received ablation results: {layer_drop.shape} ndarray")

Received ablation results: (24,) ndarray


## Examine Layer Ablation Result

In [9]:
# print drops
for layer in range(len(layer_drop)):
    print(f"Ablated Layer {layer:2d} -> Logit Drop {layer_drop[layer]:2f}")

Ablated Layer  0 -> Logit Drop 6.681055
Ablated Layer  1 -> Logit Drop 2.045313
Ablated Layer  2 -> Logit Drop 3.137500
Ablated Layer  3 -> Logit Drop 0.649219
Ablated Layer  4 -> Logit Drop 0.993750
Ablated Layer  5 -> Logit Drop 1.750781
Ablated Layer  6 -> Logit Drop 1.442969
Ablated Layer  7 -> Logit Drop 0.825000
Ablated Layer  8 -> Logit Drop 0.484375
Ablated Layer  9 -> Logit Drop 0.189062
Ablated Layer 10 -> Logit Drop 0.300000
Ablated Layer 11 -> Logit Drop -0.093750
Ablated Layer 12 -> Logit Drop 1.392187
Ablated Layer 13 -> Logit Drop 0.039062
Ablated Layer 14 -> Logit Drop -0.295312
Ablated Layer 15 -> Logit Drop 0.931250
Ablated Layer 16 -> Logit Drop 0.386719
Ablated Layer 17 -> Logit Drop -0.353125
Ablated Layer 18 -> Logit Drop 1.478125
Ablated Layer 19 -> Logit Drop -1.537500
Ablated Layer 20 -> Logit Drop -1.118750
Ablated Layer 21 -> Logit Drop 4.308594
Ablated Layer 22 -> Logit Drop 0.398438
Ablated Layer 23 -> Logit Drop 0.245312
